In [1]:
import os

for name in os.listdir("/kaggle/input/datasets/zlemglsmklkaya"):
    print(name)

caai-models-nb1-complete
caai-models-cic-scenario-t
caai-revision-splits-new
caai-revision-splits


In [2]:
import os, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

SPLITS_ROOT   = "/kaggle/input/datasets/zlemglsmklkaya/caai-revision-splits"
NB1_ROOT      = "/kaggle/input/datasets/zlemglsmklkaya/caai-models-nb1-complete"
OLD_CC_ROOT   = "/kaggle/input/datasets/zlemglsmklkaya/caai-revision-splits-new"   # ae_credit_card ONLY
AE_T_ROOT     = "/kaggle/input/datasets/zlemglsmklkaya/caai-models-cic-scenario-t"  # ae_cic_scenario_T ONLY
OUTPUT_DIR = "/kaggle/working/ig_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- Data ----
cc_dir = os.path.join(SPLITS_ROOT, "credit_card")
cc_id  = pd.read_parquet(os.path.join(cc_dir, "id.parquet"))
cc_id2 = pd.read_parquet(os.path.join(cc_dir, "id2.parquet"))
cc_ood = pd.read_parquet(os.path.join(cc_dir, "ood.parquet"))
with open(os.path.join(cc_dir, "config.json")) as f:
    CC_FEATURES = json.load(f)["model_features"]
print(f"Credit card: ID={cc_id.shape}, ID2={cc_id2.shape}, OOD={cc_ood.shape}")

cic_scenarios = {}
for scen in ["scenario_T", "scenario_A"]:
    d = os.path.join(SPLITS_ROOT, "cic_ids2017", scen)
    cic_scenarios[scen] = {
        "id":  pd.read_parquet(os.path.join(d, "id.parquet")),
        "id2": pd.read_parquet(os.path.join(d, "id2.parquet")),
        "ood": pd.read_parquet(os.path.join(d, "ood.parquet")),
    }
with open(os.path.join(SPLITS_ROOT, "cic_ids2017", "config.json")) as f:
    CIC_FEATURES = json.load(f)["model_features"]
for scen, d in cic_scenarios.items():
    print(f"{scen}: ID={d['id'].shape}, ID2={d['id2'].shape}, OOD={d['ood'].shape}")

# ---- Model classes (must match training exactly) ----
class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        enc = []; prev = input_dim
        for h in hidden_dims:
            enc += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        enc += [nn.Linear(prev, latent_dim)]
        self.encoder = nn.Sequential(*enc)
        dec = []; prev = latent_dim
        for h in reversed(hidden_dims):
            dec += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        dec += [nn.Linear(prev, input_dim)]
        self.decoder = nn.Sequential(*dec)
    def forward(self, x):
        return self.decoder(self.encoder(x))

class VAE(nn.Module):
    """Corrected formulation: global scalar output variance, clamped logvar_z."""
    def __init__(self, input_dim, hidden_dims, latent_dim):
        super().__init__()
        enc = []; prev = input_dim
        for h in hidden_dims:
            enc += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        self.encoder_body = nn.Sequential(*enc)
        self.fc_mu = nn.Linear(prev, latent_dim)
        self.fc_logvar = nn.Linear(prev, latent_dim)
        dec = []; prev = latent_dim
        for h in reversed(hidden_dims):
            dec += [nn.Linear(prev, h), nn.ReLU()]; prev = h
        dec += [nn.Linear(prev, input_dim)]
        self.decoder_body = nn.Sequential(*dec)
        self.global_logvar = nn.Parameter(torch.zeros(1))
    def encode(self, x):
        h = self.encoder_body(x)
        mu = self.fc_mu(h)
        logvar = torch.clamp(self.fc_logvar(h), min=-10, max=10)
        return mu, logvar
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std
    def decode(self, z):
        return self.decoder_body(z)
    def forward(self, x):
        mu_z, logvar_z = self.encode(x)
        z = self.reparameterize(mu_z, logvar_z)
        return self.decode(z), mu_z, logvar_z


def load_and_verify_vae_config(config_path, label):
    with open(config_path) as f:
        cfg = json.load(f)
    output_var = cfg.get("architecture", {}).get("output_variance", "")
    assert "global scalar" in output_var, (
        f"{label}: config does NOT show the corrected global-scalar formulation "
        f"(found: '{output_var}'). This may be a stale pre-fix file — STOP and check the source dataset."
    )
    print(f"  ✓ {label}: verified corrected formulation ({output_var})")
    return cfg


ARCH_CC  = {"hidden_dims": [20, 14], "latent_dim": 7}
ARCH_CIC = {"hidden_dims": [48, 32], "latent_dim": 16}

print("\nLoading + verifying models...")

# AE — Credit Card (from caai-revision-splits-new)
ae_cc = Autoencoder(len(CC_FEATURES), **ARCH_CC).to(DEVICE)
ae_cc.load_state_dict(torch.load(os.path.join(OLD_CC_ROOT, "ae_credit_card", "model.pt"), map_location=DEVICE))
ae_cc.eval()
print("  ✓ ae_credit_card loaded (from caai-revision-splits-new)")

# VAE — Credit Card (from caai-models-nb1-complete — the CORRECTED one)
load_and_verify_vae_config(os.path.join(NB1_ROOT, "vae_credit_card", "config.json"), "vae_credit_card")
vae_cc = VAE(len(CC_FEATURES), **ARCH_CC).to(DEVICE)
vae_cc.load_state_dict(torch.load(os.path.join(NB1_ROOT, "vae_credit_card", "model.pt"), map_location=DEVICE))
vae_cc.eval()

# AE — scenario_T (from caai-models-cic-scenario-t)
ae_T = Autoencoder(len(CIC_FEATURES), **ARCH_CIC).to(DEVICE)
ae_T.load_state_dict(torch.load(os.path.join(AE_T_ROOT, "ae_cic_scenario_T", "model.pt"), map_location=DEVICE))
ae_T.eval()
print("  ✓ ae_cic_scenario_T loaded (from caai-models-cic-scenario-t)")

# VAE — scenario_T (from caai-models-nb1-complete)
load_and_verify_vae_config(os.path.join(NB1_ROOT, "vae_cic_scenario_T", "config.json"), "vae_cic_scenario_T")
vae_T = VAE(len(CIC_FEATURES), **ARCH_CIC).to(DEVICE)
vae_T.load_state_dict(torch.load(os.path.join(NB1_ROOT, "vae_cic_scenario_T", "model.pt"), map_location=DEVICE))
vae_T.eval()

# AE — scenario_A (from caai-models-nb1-complete)
ae_A = Autoencoder(len(CIC_FEATURES), **ARCH_CIC).to(DEVICE)
ae_A.load_state_dict(torch.load(os.path.join(NB1_ROOT, "ae_cic_scenario_A", "model.pt"), map_location=DEVICE))
ae_A.eval()
print("  ✓ ae_cic_scenario_A loaded (from caai-models-nb1-complete)")

# VAE — scenario_A (from caai-models-nb1-complete)
load_and_verify_vae_config(os.path.join(NB1_ROOT, "vae_cic_scenario_A", "config.json"), "vae_cic_scenario_A")
vae_A = VAE(len(CIC_FEATURES), **ARCH_CIC).to(DEVICE)
vae_A.load_state_dict(torch.load(os.path.join(NB1_ROOT, "vae_cic_scenario_A", "model.pt"), map_location=DEVICE))
vae_A.eval()

print("\nAll 6 models loaded and verified.")

Device: cuda
Credit card: ID=(21731, 31), ID2=(21731, 31), OOD=(139490, 31)
scenario_T: ID=(296457, 82), ID2=(296457, 82), OOD=(1017868, 82)
scenario_A: ID=(357584, 82), ID2=(357585, 82), OOD=(615433, 82)

Loading + verifying models...
  ✓ ae_credit_card loaded (from caai-revision-splits-new)
  ✓ vae_credit_card: verified corrected formulation (global scalar, shared across all features and rows)
  ✓ ae_cic_scenario_T loaded (from caai-models-cic-scenario-t)
  ✓ vae_cic_scenario_T: verified corrected formulation (global scalar, shared across all features and rows)
  ✓ ae_cic_scenario_A loaded (from caai-models-nb1-complete)
  ✓ vae_cic_scenario_A: verified corrected formulation (global scalar, shared across all features and rows)

All 6 models loaded and verified.


In [3]:
try:
    import captum
    print("captum version:", captum.__version__)
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "captum", "--quiet"])
    import captum
    print("captum installed, version:", captum.__version__)

from captum.attr import IntegratedGradients

# Test on the Autoencoder — attribution target is reconstruction error (a scalar per instance)
def ae_anomaly_score_fn(x, model):
    recon = model(x)
    return ((recon - x) ** 2).mean(dim=1, keepdim=True)

test_instance = torch.tensor(cc_id[CC_FEATURES].iloc[[0]].values, dtype=torch.float32).to(DEVICE)
baseline = torch.zeros_like(test_instance)  # zero baseline in standardized feature space = the "average" row

ig = IntegratedGradients(lambda x: ae_anomaly_score_fn(x, ae_cc))
attributions, delta = ig.attribute(test_instance, baseline, return_convergence_delta=True)

print("Attribution shape:", attributions.shape)
print("Attribution values:", attributions.cpu().detach().numpy().flatten())
print("Convergence delta:", delta.item(), "(should be very close to 0 — this is IG's built-in correctness check)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 9.4 MB/s eta 0:00:00
captum installed, version: 0.9.0
Attribution shape: torch.Size([1, 29])
Attribution values: [-0.04129121  0.0029129   0.07149225 -0.09777121 -0.00421578 -0.06909104
 -0.0428191   0.02290399  0.00369784  0.03460367 -0.02583328  0.00037806
  0.0278493   0.00150304  0.00180744 -0.03031961  0.06845225  0.02660986
 -0.00134504  0.0011605   0.01093913 -0.01187607  0.00264506  0.00879546
  0.06420936  0.00346479 -0.001165    0.00321276 -0.00487525]
Convergence delta: 0.00011386722326278687 (should be very close to 0 — this is IG's built-in correctness check)


In [4]:
def compute_ig_attributions(model, score_fn, df, feature_cols, device, n_instances=200, seed=SEED, batch_size=64):
    """Stratified sample (up to n_instances) from df, IG attribution for each, batched for speed."""
    rng = np.random.RandomState(seed)
    n_available = len(df)
    n_sample = min(n_instances, n_available)
    idx = rng.choice(n_available, size=n_sample, replace=False)
    sample_df = df.iloc[idx]

    X = torch.tensor(sample_df[feature_cols].values, dtype=torch.float32).to(device)
    baseline = torch.zeros_like(X)

    ig = IntegratedGradients(lambda x: score_fn(x, model))

    all_attrs = []
    all_deltas = []
    for i in range(0, len(X), batch_size):
        batch = X[i:i+batch_size]
        base_batch = baseline[i:i+batch_size]
        attrs, delta = ig.attribute(batch, base_batch, return_convergence_delta=True)
        all_attrs.append(attrs.cpu().detach().numpy())
        all_deltas.append(delta.cpu().detach().numpy())

    attrs = np.concatenate(all_attrs, axis=0)
    deltas = np.concatenate(all_deltas, axis=0)
    return attrs, deltas, sample_df["Class"].values, idx


def run_ig_for_split(model, score_fn, df, feature_cols, device, split_name, n_per_class=200):
    """Stratify by class so both benign and anomaly get proper representation,
    matching how stability/faithfulness are computed per-class in the manuscript."""
    results = {}
    for cls in [0, 1]:
        class_df = df[df["Class"] == cls]
        if len(class_df) == 0:
            print(f"  {split_name} Class={cls}: no instances, skipping")
            continue
        attrs, deltas, classes, idx = compute_ig_attributions(
            model, score_fn, class_df, feature_cols, device, n_instances=n_per_class
        )
        max_abs_delta = np.abs(deltas).max()
        results[cls] = {"attributions": attrs, "row_indices": idx, "max_convergence_delta": max_abs_delta}
        print(f"  {split_name} Class={cls}: {attrs.shape[0]} instances attributed, "
              f"max|convergence_delta|={max_abs_delta:.6f}")
    return results


# Test run: AE, Credit Card, ID split only — confirm the pipeline before running everywhere
print("=== Test: AE, Credit Card, ID split ===")
test_results = run_ig_for_split(ae_cc, ae_anomaly_score_fn, cc_id, CC_FEATURES, DEVICE, "ID", n_per_class=200)

=== Test: AE, Credit Card, ID split ===
  ID Class=0: 200 instances attributed, max|convergence_delta|=0.591303
  ID Class=1: 136 instances attributed, max|convergence_delta|=0.468106


In [5]:
def compute_ig_with_full_diagnostics(model, score_fn, df, feature_cols, device, n_instances=200, seed=SEED, batch_size=64, n_steps=50):
    rng = np.random.RandomState(seed)
    n_sample = min(n_instances, len(df))
    idx = rng.choice(len(df), size=n_sample, replace=False)
    sample_df = df.iloc[idx]

    X = torch.tensor(sample_df[feature_cols].values, dtype=torch.float32).to(device)
    baseline = torch.zeros_like(X)
    ig = IntegratedGradients(lambda x: score_fn(x, model))

    all_attrs, all_deltas = [], []
    for i in range(0, len(X), batch_size):
        batch = X[i:i+batch_size]
        base_batch = baseline[i:i+batch_size]
        attrs, delta = ig.attribute(batch, base_batch, n_steps=n_steps, return_convergence_delta=True)
        all_attrs.append(attrs.cpu().detach().numpy())
        all_deltas.append(delta.cpu().detach().numpy())

    attrs = np.concatenate(all_attrs, axis=0)
    deltas = np.concatenate(all_deltas, axis=0)
    dist_from_baseline = np.linalg.norm(X.cpu().numpy(), axis=1)  # baseline is 0, so this is just ||x||
    scores_at_x = score_fn(X, model).detach().cpu().numpy().flatten()
    return attrs, deltas, dist_from_baseline, scores_at_x


attrs, deltas, dists, scores = compute_ig_with_full_diagnostics(
    ae_cc, ae_anomaly_score_fn, cc_id, CC_FEATURES, DEVICE, n_instances=336, n_steps=50
)

print("Convergence delta distribution (n_steps=50):")
print(f"  p50={np.percentile(np.abs(deltas),50):.6f}  p90={np.percentile(np.abs(deltas),90):.6f}  "
      f"p99={np.percentile(np.abs(deltas),99):.6f}  max={np.abs(deltas).max():.6f}")
print(f"\nRelative delta (|delta| / |score|) — the metric that actually matters:")
rel_delta = np.abs(deltas) / (np.abs(scores) + 1e-8)
print(f"  p50={np.percentile(rel_delta,50):.4f}  p90={np.percentile(rel_delta,90):.4f}  "
      f"p99={np.percentile(rel_delta,99):.4f}  max={rel_delta.max():.4f}")

print(f"\nCorrelation between |delta| and distance-from-baseline: "
      f"{np.corrcoef(np.abs(deltas), dists)[0,1]:.3f}")
print("(High correlation would confirm: instances far from baseline need more integration steps)")

worst_idx = np.argsort(np.abs(deltas))[-5:][::-1]
print("\n5 worst instances:")
for i in worst_idx:
    print(f"  delta={deltas[i]:.4f}  score={scores[i]:.4f}  ||x||={dists[i]:.2f}")

Convergence delta distribution (n_steps=50):
  p50=0.007353  p90=0.215732  p99=0.387886  max=0.828859

Relative delta (|delta| / |score|) — the metric that actually matters:
  p50=0.0321  p90=1.2009  p99=4.4566  max=6.3852

Correlation between |delta| and distance-from-baseline: 0.065
(High correlation would confirm: instances far from baseline need more integration steps)

5 worst instances:
  delta=-0.8289  score=0.2415  ||x||=6.01
  delta=0.6180  score=0.8266  ||x||=13.16
  delta=-0.5378  score=0.1389  ||x||=5.67
  delta=0.3879  score=0.0665  ||x||=5.45
  delta=0.3879  score=0.0665  ||x||=5.45


In [6]:
for n_steps in [50, 200, 500]:
    attrs, deltas, dists, scores = compute_ig_with_full_diagnostics(
        ae_cc, ae_anomaly_score_fn, cc_id, CC_FEATURES, DEVICE, n_instances=336, n_steps=n_steps
    )
    rel_delta = np.abs(deltas) / (np.abs(scores) + 1e-8)
    print(f"n_steps={n_steps:4d}:  delta p50={np.percentile(np.abs(deltas),50):.6f}  "
          f"p99={np.percentile(np.abs(deltas),99):.6f}  max={np.abs(deltas).max():.6f}  |  "
          f"rel_delta p50={np.percentile(rel_delta,50):.4f}  p99={np.percentile(rel_delta,99):.4f}  max={rel_delta.max():.4f}")

n_steps=  50:  delta p50=0.007353  p99=0.387886  max=0.828859  |  rel_delta p50=0.0321  p99=4.4566  max=6.3852
n_steps= 200:  delta p50=0.002059  p99=0.166999  max=0.224410  |  rel_delta p50=0.0107  p99=1.7214  max=2.6354
n_steps= 500:  delta p50=0.000729  p99=0.046551  max=0.073642  |  rel_delta p50=0.0034  p99=0.4815  max=0.8431


In [7]:
attrs, deltas, dists, scores = compute_ig_with_full_diagnostics(
    ae_cc, ae_anomaly_score_fn, cc_id, CC_FEATURES, DEVICE, n_instances=336, n_steps=300
)
rel_delta = np.abs(deltas) / (np.abs(scores) + 1e-8)

for threshold in [0.1, 0.2, 0.5]:
    n_excluded = (rel_delta > threshold).sum()
    print(f"Threshold {threshold}: {n_excluded}/{len(rel_delta)} excluded ({100*n_excluded/len(rel_delta):.1f}%)")

print(f"\nAt n_steps=300: rel_delta p50={np.percentile(rel_delta,50):.4f}  p90={np.percentile(rel_delta,90):.4f}  "
      f"p99={np.percentile(rel_delta,99):.4f}  max={rel_delta.max():.4f}")

Threshold 0.1: 43/336 excluded (12.8%)
Threshold 0.2: 26/336 excluded (7.7%)
Threshold 0.5: 10/336 excluded (3.0%)

At n_steps=300: rel_delta p50=0.0066  p90=0.1375  p99=0.8608  max=1.4461


In [8]:
IG_CONFIG = {
    "n_steps": 300,
    "exclusion_threshold": 0.2,  # relative convergence delta; instances above this are excluded, not attributed
    "n_instances_target": 200,   # after exclusion — oversample to compensate
    "n_instances_oversample": 240,
}
print(json.dumps(IG_CONFIG, indent=2))


def run_ig_production(model, score_fn, df, feature_cols, device, split_name, config=IG_CONFIG, seed=SEED):
    results = {}
    for cls in [0, 1]:
        class_df = df[df["Class"] == cls]
        if len(class_df) == 0:
            print(f"  {split_name} Class={cls}: no instances, skipping")
            continue

        n_sample = min(config["n_instances_oversample"], len(class_df))
        attrs, deltas, dists, scores = compute_ig_with_full_diagnostics(
            model, score_fn, class_df, feature_cols, device, n_instances=n_sample, n_steps=config["n_steps"]
        )
        rel_delta = np.abs(deltas) / (np.abs(scores) + 1e-8)
        keep_mask = rel_delta <= config["exclusion_threshold"]

        n_excluded = (~keep_mask).sum()
        kept_attrs = attrs[keep_mask]
        if len(kept_attrs) > config["n_instances_target"]:
            kept_attrs = kept_attrs[:config["n_instances_target"]]

        results[cls] = {
            "attributions": kept_attrs,
            "n_attempted": n_sample,
            "n_excluded": int(n_excluded),
            "n_final": len(kept_attrs),
            "exclusion_rate": float(n_excluded / n_sample),
        }
        print(f"  {split_name} Class={cls}: {n_sample} attempted, {n_excluded} excluded "
              f"({100*n_excluded/n_sample:.1f}%), {len(kept_attrs)} final")
    return results


print("\n=== Production validation: AE, Credit Card, ID split ===")
ae_cc_id_ig = run_ig_production(ae_cc, ae_anomaly_score_fn, cc_id, CC_FEATURES, DEVICE, "ID")

{
  "n_steps": 300,
  "exclusion_threshold": 0.2,
  "n_instances_target": 200,
  "n_instances_oversample": 240
}

=== Production validation: AE, Credit Card, ID split ===
  ID Class=0: 240 attempted, 15 excluded (6.2%), 200 final
  ID Class=1: 136 attempted, 0 excluded (0.0%), 136 final


In [9]:
def vae_anomaly_score_fn(x, model):
    """Deterministic NLL using the encoder MEAN (no sampling) — required for IG,
    which needs a differentiable, deterministic function. This differs from NB1's
    scoring (Monte Carlo average over 10 stochastic draws) by design; documented
    as a methodological note for the reproducibility appendix."""
    mu_z, _ = model.encode(x)
    out_mu = model.decode(mu_z)
    logvar = torch.clamp(model.global_logvar, min=-10, max=10)
    nll = 0.5 * (logvar + ((x - out_mu) ** 2) / torch.exp(logvar) + np.log(2 * np.pi))
    return nll.sum(dim=1, keepdim=True)


print("=== Production validation: VAE, Credit Card, ID split ===")
vae_cc_id_ig = run_ig_production(vae_cc, vae_anomaly_score_fn, cc_id, CC_FEATURES, DEVICE, "ID")

=== Production validation: VAE, Credit Card, ID split ===
  ID Class=0: 240 attempted, 0 excluded (0.0%), 200 final
  ID Class=1: 136 attempted, 0 excluded (0.0%), 136 final


In [10]:
def run_full_ig_for_model(model, score_fn, splits_dict, feature_cols, device, model_label, config=IG_CONFIG):
    all_results = {}
    for split_name, split_df in splits_dict.items():
        print(f"\n--- {model_label} — {split_name} ---")
        all_results[split_name] = run_ig_production(model, score_fn, split_df, feature_cols, device, split_name, config)
    return all_results


cc_splits = {"id": cc_id, "id2": cc_id2, "ood": cc_ood}

print("========== AE — Credit Card ==========")
ae_cc_ig = run_full_ig_for_model(ae_cc, ae_anomaly_score_fn, cc_splits, CC_FEATURES, DEVICE, "AE-CC")

print("\n\n========== VAE — Credit Card ==========")
vae_cc_ig = run_full_ig_for_model(vae_cc, vae_anomaly_score_fn, cc_splits, CC_FEATURES, DEVICE, "VAE-CC")

========== AE — Credit Card ==========

--- AE-CC — id ---
  id Class=0: 240 attempted, 15 excluded (6.2%), 200 final
  id Class=1: 136 attempted, 0 excluded (0.0%), 136 final

--- AE-CC — id2 ---
  id2 Class=0: 240 attempted, 26 excluded (10.8%), 200 final
  id2 Class=1: 136 attempted, 0 excluded (0.0%), 136 final

--- AE-CC — ood ---
  ood Class=0: 240 attempted, 1 excluded (0.4%), 200 final
  ood Class=1: 201 attempted, 0 excluded (0.0%), 200 final


========== VAE — Credit Card ==========

--- VAE-CC — id ---
  id Class=0: 240 attempted, 0 excluded (0.0%), 200 final
  id Class=1: 136 attempted, 0 excluded (0.0%), 136 final

--- VAE-CC — id2 ---
  id2 Class=0: 240 attempted, 0 excluded (0.0%), 200 final
  id2 Class=1: 136 attempted, 0 excluded (0.0%), 136 final

--- VAE-CC — ood ---
  ood Class=0: 240 attempted, 0 excluded (0.0%), 200 final
  ood Class=1: 201 attempted, 0 excluded (0.0%), 200 final


In [11]:
def save_ig_results(results_by_split, model_label, feature_cols, output_dir):
    model_dir = os.path.join(output_dir, f"ig_{model_label}")
    os.makedirs(model_dir, exist_ok=True)

    summary = {}
    for split_name, split_results in results_by_split.items():
        for cls, res in split_results.items():
            key = f"{split_name}_class{cls}"
            np.save(os.path.join(model_dir, f"attributions_{key}.npy"), res["attributions"])
            summary[key] = {
                "n_attempted": res["n_attempted"], "n_excluded": res["n_excluded"],
                "n_final": res["n_final"], "exclusion_rate": res["exclusion_rate"],
            }

    config_out = {
        "model": model_label, "feature_cols": feature_cols,
        "ig_config": IG_CONFIG, "exclusion_summary": summary,
    }
    with open(os.path.join(model_dir, "config.json"), "w") as f:
        json.dump(config_out, f, indent=2)
    print(f"Saved: {model_dir}")
    print(os.listdir(model_dir))


save_ig_results(ae_cc_ig, "ae_credit_card", CC_FEATURES, OUTPUT_DIR)
save_ig_results(vae_cc_ig, "vae_credit_card", CC_FEATURES, OUTPUT_DIR)

Saved: /kaggle/working/ig_results/ig_ae_credit_card
['attributions_ood_class1.npy', 'config.json', 'attributions_id2_class1.npy', 'attributions_ood_class0.npy', 'attributions_id2_class0.npy', 'attributions_id_class0.npy', 'attributions_id_class1.npy']
Saved: /kaggle/working/ig_results/ig_vae_credit_card
['attributions_ood_class1.npy', 'config.json', 'attributions_id2_class1.npy', 'attributions_ood_class0.npy', 'attributions_id2_class0.npy', 'attributions_id_class0.npy', 'attributions_id_class1.npy']


In [12]:
T_id  = cic_scenarios["scenario_T"]["id"]
T_id2 = cic_scenarios["scenario_T"]["id2"]
T_ood = cic_scenarios["scenario_T"]["ood"]
T_splits = {"id": T_id, "id2": T_id2, "ood": T_ood}

print("========== AE — scenario_T ==========")
ae_T_ig = run_full_ig_for_model(ae_T, ae_anomaly_score_fn, T_splits, CIC_FEATURES, DEVICE, "AE-T")

========== AE — scenario_T ==========

--- AE-T — id ---
  id Class=0: 240 attempted, 121 excluded (50.4%), 119 final
  id Class=1: 240 attempted, 10 excluded (4.2%), 200 final

--- AE-T — id2 ---
  id2 Class=0: 240 attempted, 124 excluded (51.7%), 116 final
  id2 Class=1: 240 attempted, 15 excluded (6.2%), 200 final

--- AE-T — ood ---
  ood Class=0: 240 attempted, 117 excluded (48.8%), 123 final
  ood Class=1: 240 attempted, 1 excluded (0.4%), 200 final


In [13]:
def diagnose_exclusions(model, score_fn, df, feature_cols, device, n_instances=240, n_steps=300):
    attrs, deltas, dists, scores = compute_ig_with_full_diagnostics(
        model, score_fn, df, feature_cols, device, n_instances=n_instances, n_steps=n_steps
    )
    rel_delta = np.abs(deltas) / (np.abs(scores) + 1e-8)
    excluded_mask = rel_delta > 0.2

    print(f"Excluded instances' score percentile within the full sample:")
    score_ranks = pd.Series(scores).rank(pct=True).values
    print(f"  Excluded (n={excluded_mask.sum()}): mean score percentile = {score_ranks[excluded_mask].mean():.3f}")
    print(f"  Kept (n={(~excluded_mask).sum()}): mean score percentile = {score_ranks[~excluded_mask].mean():.3f}")
    print(f"\n  Excluded scores: min={scores[excluded_mask].min():.4f}, "
          f"median={np.median(scores[excluded_mask]):.4f}, max={scores[excluded_mask].max():.4f}")
    print(f"  Kept scores: min={scores[~excluded_mask].min():.4f}, "
          f"median={np.median(scores[~excluded_mask]):.4f}, max={scores[~excluded_mask].max():.4f}")


benign_T_id = T_id[T_id["Class"] == 0]
diagnose_exclusions(ae_T, ae_anomaly_score_fn, benign_T_id, CIC_FEATURES, DEVICE)

Excluded instances' score percentile within the full sample:
  Excluded (n=121): mean score percentile = 0.304
  Kept (n=119): mean score percentile = 0.704

  Excluded scores: min=0.0001, median=0.0003, max=0.0165
  Kept scores: min=0.0002, median=0.0027, max=0.6258


In [14]:
attrs, deltas, dists, scores = compute_ig_with_full_diagnostics(
    ae_T, ae_anomaly_score_fn, benign_T_id, CIC_FEATURES, DEVICE, n_instances=240, n_steps=300
)
rel_delta = np.abs(deltas) / (np.abs(scores) + 1e-8)
excluded_mask = rel_delta > 0.2

print(f"Absolute delta — excluded (n={excluded_mask.sum()}): "
      f"median={np.median(np.abs(deltas[excluded_mask])):.6f}, max={np.abs(deltas[excluded_mask]).max():.6f}")
print(f"Absolute delta — kept (n={(~excluded_mask).sum()}): "
      f"median={np.median(np.abs(deltas[~excluded_mask])):.6f}, max={np.abs(deltas[~excluded_mask]).max():.6f}")

Absolute delta — excluded (n=121): median=0.000329, max=0.004538
Absolute delta — kept (n=119): median=0.000038, max=0.001468


In [15]:
def combined_exclusion_test(deltas, scores, rel_threshold=0.2, abs_floors=[0.0005, 0.001, 0.002, 0.005]):
    rel_delta = np.abs(deltas) / (np.abs(scores) + 1e-8)
    print(f"Relative-only (current): {(rel_delta > rel_threshold).sum()}/{len(deltas)} excluded")
    for floor in abs_floors:
        combined_mask = (rel_delta > rel_threshold) & (np.abs(deltas) > floor)
        print(f"  + abs_delta > {floor}: {combined_mask.sum()}/{len(deltas)} excluded")

combined_exclusion_test(deltas, scores)

Relative-only (current): 121/240 excluded
  + abs_delta > 0.0005: 36/240 excluded
  + abs_delta > 0.001: 12/240 excluded
  + abs_delta > 0.002: 3/240 excluded
  + abs_delta > 0.005: 0/240 excluded


In [16]:
IG_CONFIG["abs_delta_floor"] = 0.001
IG_CONFIG["exclusion_logic"] = "exclude if (rel_delta > exclusion_threshold) AND (abs_delta > abs_delta_floor) — avoids penalizing near-zero-score rows for numerically trivial absolute error"
print(json.dumps(IG_CONFIG, indent=2))


def run_ig_production(model, score_fn, df, feature_cols, device, split_name, config=IG_CONFIG, seed=SEED):
    results = {}
    for cls in [0, 1]:
        class_df = df[df["Class"] == cls]
        if len(class_df) == 0:
            print(f"  {split_name} Class={cls}: no instances, skipping")
            continue

        n_sample = min(config["n_instances_oversample"], len(class_df))
        attrs, deltas, dists, scores = compute_ig_with_full_diagnostics(
            model, score_fn, class_df, feature_cols, device, n_instances=n_sample, n_steps=config["n_steps"]
        )
        rel_delta = np.abs(deltas) / (np.abs(scores) + 1e-8)
        exclude_mask = (rel_delta > config["exclusion_threshold"]) & (np.abs(deltas) > config["abs_delta_floor"])
        keep_mask = ~exclude_mask

        n_excluded = exclude_mask.sum()
        kept_attrs = attrs[keep_mask]
        if len(kept_attrs) > config["n_instances_target"]:
            kept_attrs = kept_attrs[:config["n_instances_target"]]

        results[cls] = {
            "attributions": kept_attrs, "n_attempted": n_sample, "n_excluded": int(n_excluded),
            "n_final": len(kept_attrs), "exclusion_rate": float(n_excluded / n_sample),
        }
        print(f"  {split_name} Class={cls}: {n_sample} attempted, {n_excluded} excluded "
              f"({100*n_excluded/n_sample:.1f}%), {len(kept_attrs)} final")
    return results


print("=== Re-validation: AE, scenario_T, all splits (combined criterion) ===")
ae_T_ig = run_full_ig_for_model(ae_T, ae_anomaly_score_fn, T_splits, CIC_FEATURES, DEVICE, "AE-T")

{
  "n_steps": 300,
  "exclusion_threshold": 0.2,
  "n_instances_target": 200,
  "n_instances_oversample": 240,
  "abs_delta_floor": 0.001,
  "exclusion_logic": "exclude if (rel_delta > exclusion_threshold) AND (abs_delta > abs_delta_floor) \u2014 avoids penalizing near-zero-score rows for numerically trivial absolute error"
}
=== Re-validation: AE, scenario_T, all splits (combined criterion) ===

--- AE-T — id ---
  id Class=0: 240 attempted, 12 excluded (5.0%), 200 final
  id Class=1: 240 attempted, 0 excluded (0.0%), 200 final

--- AE-T — id2 ---
  id2 Class=0: 240 attempted, 12 excluded (5.0%), 200 final
  id2 Class=1: 240 attempted, 2 excluded (0.8%), 200 final

--- AE-T — ood ---
  ood Class=0: 240 attempted, 6 excluded (2.5%), 200 final
  ood Class=1: 240 attempted, 0 excluded (0.0%), 200 final


In [17]:
print("========== VAE — scenario_T ==========")
vae_T_ig = run_full_ig_for_model(vae_T, vae_anomaly_score_fn, T_splits, CIC_FEATURES, DEVICE, "VAE-T")

print("\n\n========== Re-run: AE — Credit Card (combined criterion) ==========")
ae_cc_ig = run_full_ig_for_model(ae_cc, ae_anomaly_score_fn, cc_splits, CC_FEATURES, DEVICE, "AE-CC")

print("\n\n========== Re-run: VAE — Credit Card (combined criterion) ==========")
vae_cc_ig = run_full_ig_for_model(vae_cc, vae_anomaly_score_fn, cc_splits, CC_FEATURES, DEVICE, "VAE-CC")

========== VAE — scenario_T ==========

--- VAE-T — id ---
  id Class=0: 240 attempted, 4 excluded (1.7%), 200 final
  id Class=1: 240 attempted, 0 excluded (0.0%), 200 final

--- VAE-T — id2 ---
  id2 Class=0: 240 attempted, 2 excluded (0.8%), 200 final
  id2 Class=1: 240 attempted, 1 excluded (0.4%), 200 final

--- VAE-T — ood ---
  ood Class=0: 240 attempted, 2 excluded (0.8%), 200 final
  ood Class=1: 240 attempted, 0 excluded (0.0%), 200 final


========== Re-run: AE — Credit Card (combined criterion) ==========

--- AE-CC — id ---
  id Class=0: 240 attempted, 15 excluded (6.2%), 200 final
  id Class=1: 136 attempted, 0 excluded (0.0%), 136 final

--- AE-CC — id2 ---
  id2 Class=0: 240 attempted, 26 excluded (10.8%), 200 final
  id2 Class=1: 136 attempted, 0 excluded (0.0%), 136 final

--- AE-CC — ood ---
  ood Class=0: 240 attempted, 1 excluded (0.4%), 200 final
  ood Class=1: 201 attempted, 0 excluded (0.0%), 200 final


========== Re-run: VAE — Credit Card (combined criterion) 

In [18]:
save_ig_results(ae_T_ig, "ae_cic_scenario_T", CIC_FEATURES, OUTPUT_DIR)
save_ig_results(vae_T_ig, "vae_cic_scenario_T", CIC_FEATURES, OUTPUT_DIR)
save_ig_results(ae_cc_ig, "ae_credit_card", CC_FEATURES, OUTPUT_DIR)   # overwrites the earlier relative-only-criterion save
save_ig_results(vae_cc_ig, "vae_credit_card", CC_FEATURES, OUTPUT_DIR)

Saved: /kaggle/working/ig_results/ig_ae_cic_scenario_T
['attributions_ood_class1.npy', 'config.json', 'attributions_id2_class1.npy', 'attributions_ood_class0.npy', 'attributions_id2_class0.npy', 'attributions_id_class0.npy', 'attributions_id_class1.npy']
Saved: /kaggle/working/ig_results/ig_vae_cic_scenario_T
['attributions_ood_class1.npy', 'config.json', 'attributions_id2_class1.npy', 'attributions_ood_class0.npy', 'attributions_id2_class0.npy', 'attributions_id_class0.npy', 'attributions_id_class1.npy']
Saved: /kaggle/working/ig_results/ig_ae_credit_card
['attributions_ood_class1.npy', 'config.json', 'attributions_id2_class1.npy', 'attributions_ood_class0.npy', 'attributions_id2_class0.npy', 'attributions_id_class0.npy', 'attributions_id_class1.npy']
Saved: /kaggle/working/ig_results/ig_vae_credit_card
['attributions_ood_class1.npy', 'config.json', 'attributions_id2_class1.npy', 'attributions_ood_class0.npy', 'attributions_id2_class0.npy', 'attributions_id_class0.npy', 'attributions_

In [19]:
A_id  = cic_scenarios["scenario_A"]["id"]
A_id2 = cic_scenarios["scenario_A"]["id2"]
A_ood = cic_scenarios["scenario_A"]["ood"]
A_splits = {"id": A_id, "id2": A_id2, "ood": A_ood}

print("========== AE — scenario_A ==========")
ae_A_ig = run_full_ig_for_model(ae_A, ae_anomaly_score_fn, A_splits, CIC_FEATURES, DEVICE, "AE-A")

print("\n\n========== VAE — scenario_A ==========")
vae_A_ig = run_full_ig_for_model(vae_A, vae_anomaly_score_fn, A_splits, CIC_FEATURES, DEVICE, "VAE-A")

========== AE — scenario_A ==========

--- AE-A — id ---
  id Class=0: 240 attempted, 3 excluded (1.2%), 200 final
  id Class=1: 240 attempted, 2 excluded (0.8%), 200 final

--- AE-A — id2 ---
  id2 Class=0: 240 attempted, 2 excluded (0.8%), 200 final
  id2 Class=1: 240 attempted, 4 excluded (1.7%), 200 final

--- AE-A — ood ---
  ood Class=0: 240 attempted, 3 excluded (1.2%), 200 final
  ood Class=1: 240 attempted, 0 excluded (0.0%), 200 final


========== VAE — scenario_A ==========

--- VAE-A — id ---
  id Class=0: 240 attempted, 2 excluded (0.8%), 200 final
  id Class=1: 240 attempted, 3 excluded (1.2%), 200 final

--- VAE-A — id2 ---
  id2 Class=0: 240 attempted, 3 excluded (1.2%), 200 final
  id2 Class=1: 240 attempted, 6 excluded (2.5%), 200 final

--- VAE-A — ood ---
  ood Class=0: 240 attempted, 5 excluded (2.1%), 200 final
  ood Class=1: 240 attempted, 6 excluded (2.5%), 200 final


In [20]:
save_ig_results(ae_A_ig, "ae_cic_scenario_A", CIC_FEATURES, OUTPUT_DIR)
save_ig_results(vae_A_ig, "vae_cic_scenario_A", CIC_FEATURES, OUTPUT_DIR)

Saved: /kaggle/working/ig_results/ig_ae_cic_scenario_A
['attributions_ood_class1.npy', 'config.json', 'attributions_id2_class1.npy', 'attributions_ood_class0.npy', 'attributions_id2_class0.npy', 'attributions_id_class0.npy', 'attributions_id_class1.npy']
Saved: /kaggle/working/ig_results/ig_vae_cic_scenario_A
['attributions_ood_class1.npy', 'config.json', 'attributions_id2_class1.npy', 'attributions_ood_class0.npy', 'attributions_id2_class0.npy', 'attributions_id_class0.npy', 'attributions_id_class1.npy']


In [21]:
import shutil
shutil.make_archive("/kaggle/working/ig_archive_nb4", "zip", "/kaggle/working/ig_results")
print("Size (MB):", os.path.getsize("/kaggle/working/ig_archive_nb4.zip") / (1024*1024))

Size (MB): 1.4064750671386719
